# 02 — Primary and Bootstrap Analysis

**Research question:** As Australia's material conditions improved, did perceived social support and emotional well-being deteriorate relative to comparable countries?

**Primary outcomes:** real PPP-adjusted household income per person (USD), employment rate (percentage points), lack of social support (percentage points), and negative affect (percentage points).  
**Method:** descriptive common-endpoint comparisons and same-year gaps. Countries must report both exact endpoints; lower-is-better measures are sign-oriented before ranking.  
**Success criterion:** a conclusion must be stable across the pre-specified English-speaking peer group, the broad supplied-country reference set, normal-value-only data, and leave-one-peer-out checks. This is not a causal design.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve paths from either the repository root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.oecd_audit import INDICATOR_SPECS, load_clean

ENGLISH_SPEAKING_PEERS = ['CAN', 'NZL', 'GBR', 'USA']
PRIMARY_CODES = ['1_1', '2_1', '2_7', '2_2', '1_2']
# Keep the four pre-specified outcomes and exact displayed endpoints fixed.
MATERIAL_SOCIAL_SPECS = [
    ('1_1', 2010, 2024, '2010–2024'),
    ('2_1', 2010, 2024, '2010–2024'),
    ('7_1_DEP', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
    ('11_2', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
]
TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'

# Build exact-endpoint changes so every eligible country is compared on the same period.
def endpoint_changes(data, code, references, start=2010, end=2024, normal_only=False):
    subset = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *references]) & data.year.isin([start, end])].copy()
    # This optional filter supports status-flag sensitivity checks without changing the primary data.
    if normal_only:
        subset = subset.loc[subset.status_code.eq('A')]
    # Keep one row per independent period so pooled social windows are not double-counted.
    subset = subset.sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    wide = subset.pivot(index='country_code', columns='year', values='value').reindex(columns=[start, end]).dropna().reset_index()
    wide = wide.rename(columns={start: 'start_value', end: 'end_value'})
    # Preserve native changes while orienting lower-is-better outcomes for fair ranking.
    wide['absolute_change'] = wide.end_value - wide.start_value
    wide['oriented_change'] = wide.absolute_change * (1 if INDICATOR_SPECS[code].direction == 'higher' else -1)
    return wide


def material_social_endpoints(data):
    """Build the four-outcome table using only exact common endpoints."""
    # Start with all supplied countries, then let exact endpoint availability define comparators.
    all_references = sorted(set(data.country_code) - {'AUS'})
    rows = []
    for code, start, end, period_label in MATERIAL_SOCIAL_SPECS:
        changes = endpoint_changes(data, code, all_references, start, end)
        if 'AUS' not in changes.country_code.values:
            raise ValueError(f'Australia lacks a required endpoint for {code}.')
        australia = changes.loc[changes.country_code.eq('AUS')].iloc[0]
        australia_raw = data.loc[
            data.country_code.eq('AUS') & data.indicator_code.eq(code) & data.year.isin([start, end])
        ].sort_values('year')
        if australia_raw.year.tolist() != [start, end]:
            raise ValueError(f'Australia does not have both displayed endpoints for {code}.')
        comparators = changes.loc[changes.country_code.ne('AUS')]
        comparator_count = int(len(comparators))
        australia_rank = changes.oriented_change.rank(ascending=False, method='average').loc[australia.name]
        total_country_count = len(changes)
        rows.append({
            'indicator_code': code,
            'indicator': data.loc[data.indicator_code.eq(code), 'indicator'].iloc[0],
            'unit': data.loc[data.indicator_code.eq(code), 'unit'].iloc[0],
            'comparison_period': period_label,
            'start_year_displayed': start,
            'end_year_displayed': end,
            'start_independent_period': australia_raw.iloc[0].independent_period,
            'end_independent_period': australia_raw.iloc[1].independent_period,
            'better_direction': INDICATOR_SPECS[code].direction,
            'australia_start_value': australia.start_value,
            'australia_end_value': australia.end_value,
            'australia_native_change': australia.absolute_change,
            'native_change_definition': 'end value minus start value; natural unit and sign retained',
            'comparator_median_native_change': comparators.absolute_change.median(),
            'australia_minus_comparator_median_oriented': australia.oriented_change - comparators.oriented_change.median(),
            'oriented_gap_definition': 'positive means more favourable Australian change',
            'australia_favourable_percentile': 100 * (total_country_count - australia_rank) / (total_country_count - 1),
            'eligible_comparator_country_count': comparator_count,
        })
    return pd.DataFrame(rows)


# Recalculate Australia directly from tidy data to guard against workflow drift.
def direct_australia_endpoint_audit(data):
    """Independently recalculate Australian endpoints without endpoint_changes."""
    rows = []
    for code, start, end, _ in MATERIAL_SOCIAL_SPECS:
        direct = data.loc[
            data.country_code.eq('AUS') & data.indicator_code.eq(code) & data.year.isin([start, end])
        ].sort_values('year')
        if direct.year.tolist() != [start, end]:
            raise ValueError(f'Direct audit failed to find both Australian endpoints for {code}.')
        rows.append({
            'indicator_code': code,
            'direct_start_value': direct.iloc[0].value,
            'direct_end_value': direct.iloc[1].value,
            'direct_native_change': direct.iloc[1].value - direct.iloc[0].value,
            'direct_start_period': direct.iloc[0].independent_period,
            'direct_end_period': direct.iloc[1].independent_period,
        })
    return pd.DataFrame(rows)

def comparative_scorecard(data, codes=PRIMARY_CODES, references=ENGLISH_SPEAKING_PEERS, start=2010, end=2024, normal_only=False):
    rows = []
    for code in codes:
        changes = endpoint_changes(data, code, references, start, end, normal_only)
        aus, peers = changes.loc[changes.country_code.eq('AUS')], changes.loc[changes.country_code.ne('AUS')]
        if aus.empty or peers.empty: continue
        au = aus.iloc[0]; rank = changes.oriented_change.rank(ascending=False, method='average').loc[aus.index[0]]; n = len(changes)
        rows.append({'indicator_code': code, 'indicator': data.loc[data.indicator_code.eq(code), 'indicator'].iloc[0], 'start_year': start, 'end_year': end, 'australia_start_value': au.start_value, 'australia_end_value': au.end_value, 'australia_absolute_change': au.absolute_change, 'australia_oriented_change': au.oriented_change, 'reference_country_count': len(peers), 'reference_median_oriented_change': peers.oriented_change.median(), 'australia_minus_reference_median': au.oriented_change - peers.oriented_change.median(), 'australia_change_percentile': 100 if n == 1 else 100 * (n-rank)/(n-1), 'normal_values_only': normal_only})
    return pd.DataFrame(rows)

def relative_trend(data, code, references, start=2010, end=2024):
    subset = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *references]) & data.year.between(start, end)].sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    aus = subset.loc[subset.country_code.eq('AUS'), ['year', 'value']].rename(columns={'value': 'australia_value'})
    peer = subset.loc[subset.country_code.ne('AUS')].groupby('year', as_index=False).agg(reference_median=('value', 'median'), reference_country_count=('country_code', 'nunique'))
    out = aus.merge(peer, on='year'); out['oriented_gap'] = (out.australia_value-out.reference_median) * (1 if INDICATOR_SPECS[code].direction == 'higher' else -1)
    return out

def leave_one_out(data, codes=PRIMARY_CODES):
    results = []
    for omitted in ENGLISH_SPEAKING_PEERS:
        table = comparative_scorecard(data, codes, [p for p in ENGLISH_SPEAKING_PEERS if p != omitted])
        table.insert(0, 'omitted_peer', omitted); results.append(table)
    return pd.concat(results, ignore_index=True)

df = load_clean()
df.shape

# Method 2 — Primary Same-Endpoint Comparison

## Pre-specified descriptive baseline

This is a small country-level panel, not an experiment. We therefore report transparent effect sizes and coverage rather than p-values that would overstate precision. The primary sensitivity group—Canada, New Zealand, the United Kingdom and the United States—is interpretable but not uniquely correct; all supplied countries are the breadth check.

Method 2 starts with a single four-outcome common-endpoint workflow. A country enters an outcome-specific comparison only when it reports both of Australia's exact displayed endpoints: 2010 and 2024 for income/employment, and the displayed rows representing the 2008–10 and 2023–25 pooled social windows. The table retains native-unit changes for interpretation, then adds a direction-oriented Australia-minus-comparator-median gap and favourable percentile for cross-outcome comparison. Uncertainty and robustness are subsequent Method 2 steps.

In [ ]:
# Method 2, steps 1–3: one four-outcome workflow with exact common endpoints.
primary_endpoints = material_social_endpoints(df)
direct_endpoint_audit = direct_australia_endpoint_audit(df)

# Correctness checks: all primary outcomes are present, every reported
# Australian endpoint comes directly from the audited tidy data, and only
# countries with both exact displayed endpoints are retained as comparators.
assert primary_endpoints.indicator_code.tolist() == [spec[0] for spec in MATERIAL_SOCIAL_SPECS]
assert primary_endpoints[['australia_start_value', 'australia_end_value', 'australia_native_change']].notna().all().all()
assert primary_endpoints[['comparator_median_native_change', 'australia_minus_comparator_median_oriented', 'australia_favourable_percentile']].notna().all().all()
assert primary_endpoints.eligible_comparator_country_count.tolist() == [31, 43, 46, 46]
assert primary_endpoints.australia_favourable_percentile.between(0, 100).all()
assert primary_endpoints.native_change_definition.nunique() == 1
assert primary_endpoints.oriented_gap_definition.nunique() == 1
assert primary_endpoints.better_direction.tolist() == ['higher', 'higher', 'lower', 'lower']
expected_periods = {
    '1_1': ('2010', '2024'), '2_1': ('2010', '2024'),
    '7_1_DEP': ('2008-10', '2023-25'), '11_2': ('2008-10', '2023-25'),
}
expected_australia_endpoints = {
    '1_1': (44625.0, 50629.0), '2_1': (75.444, 80.262),
    '7_1_DEP': (4.926543, 10.043193), '11_2': (12.244020, 14.853043),
}
for row in primary_endpoints.itertuples(index=False):
    expected_start, expected_end = expected_australia_endpoints[row.indicator_code]
    assert abs(row.australia_start_value - expected_start) < 1e-5
    assert abs(row.australia_end_value - expected_end) < 1e-5
    assert (row.start_independent_period, row.end_independent_period) == expected_periods[row.indicator_code]
    common = endpoint_changes(df, row.indicator_code, sorted(set(df.country_code) - {'AUS'}), row.start_year_displayed, row.end_year_displayed)
    assert len(common) == row.eligible_comparator_country_count + 1
    comparators = common.loc[common.country_code.ne('AUS')]
    sign = 1 if INDICATOR_SPECS[row.indicator_code].direction == 'higher' else -1
    expected_gap = sign * (row.australia_native_change - comparators.absolute_change.median())
    assert abs(row.australia_minus_comparator_median_oriented - expected_gap) < 1e-10
    expected_rank = common.oriented_change.rank(ascending=False, method='average').loc[common.country_code.eq('AUS')].iloc[0]
    expected_percentile = 100 * (len(common) - expected_rank) / (len(common) - 1)
    assert abs(row.australia_favourable_percentile - expected_percentile) < 1e-10

reconciled = primary_endpoints.merge(direct_endpoint_audit, on='indicator_code', validate='one_to_one')
assert (reconciled.australia_start_value == reconciled.direct_start_value).all()
assert (reconciled.australia_end_value == reconciled.direct_end_value).all()
assert (reconciled.australia_native_change == reconciled.direct_native_change).all()
assert (reconciled.start_independent_period == reconciled.direct_start_period).all()
assert (reconciled.end_independent_period == reconciled.direct_end_period).all()

# Freeze the validated primary table and verify its CSV round trip.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
primary_results_path = TABLE_DIR / 'material_social_primary_results.csv'
primary_endpoints.to_csv(primary_results_path, index=False)
reloaded_primary_endpoints = pd.read_csv(primary_results_path)
pd.testing.assert_frame_equal(
    primary_endpoints, reloaded_primary_endpoints, check_dtype=False,
    check_exact=False, rtol=1e-12, atol=1e-12,
)

display(primary_endpoints)
display(direct_endpoint_audit)
print(f'Wrote and round-trip validated {primary_results_path.relative_to(PROJECT_ROOT)}')

# Existing economic exploration outputs retained for later sensitivity work.
english_peers = comparative_scorecard(df)
all_countries = comparative_scorecard(df, references=sorted(set(df.country_code) - {'AUS'}))
normal_only = comparative_scorecard(df, references=sorted(set(df.country_code) - {'AUS'}), normal_only=True)
inequality = comparative_scorecard(df, codes=['1_2'], references=sorted(set(df.country_code) - {'AUS'}), start=2012, end=2020)
english_peers.to_csv(TABLE_DIR / 'economic_change_scorecard_english_peers.csv', index=False)
all_countries.to_csv(TABLE_DIR / 'economic_change_scorecard_all_countries.csv', index=False)
normal_only.to_csv(TABLE_DIR / 'economic_change_scorecard_normal_values.csv', index=False)
inequality.to_csv(TABLE_DIR / 'income_inequality_change_scorecard.csv', index=False)
leave_one_out(df).to_csv(TABLE_DIR / 'economic_leave_one_out.csv', index=False)
for code, name in [('1_1', 'income'), ('2_1', 'employment'), ('2_7', 'long_hours')]:
    relative_trend(df, code, ENGLISH_SPEAKING_PEERS).to_csv(TABLE_DIR / f'{name}_relative_trend_english_peers.csv', index=False)
display(english_peers)
display(all_countries)

# Method 3 — Comparator-Composition Bootstrap and Placebo Ranking

Method 3 holds Australia's exact observed endpoint change fixed and resamples only the eligible comparator countries. Its bootstrap interval therefore measures sensitivity to comparator-country composition, not OECD survey-sampling uncertainty. The placebo rank is an alternative representation of the same endpoint comparison as Method 2, rather than independent evidence.

In [ ]:
# Fix the seed so comparator-resampling intervals reproduce exactly.
METHOD3_SEED = 20260720
METHOD3_REPLICATES = 10_000
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
OUTCOME_ORDER = [spec[0] for spec in MATERIAL_SOCIAL_SPECS]
EXPECTED_COUNTS = [31, 43, 46, 46]
EXPECTED_GAPS = [-956.0, -0.864, -4.290870, -2.662420]
UNCERTAINTY_SCOPE = ('Comparator-country composition sensitivity; not OECD survey-sampling uncertainty.')
EVIDENCE_RELATIONSHIP = ('Placebo rank and the Method 2 endpoint percentile are alternative presentations of the same comparison, not independent evidence.')

# Reuse the Method 2 exact-endpoint rule for every Method 3 outcome.
def exact_outcome_changes(code, start, end):
    references = sorted(set(df.country_code) - {'AUS'})
    changes = endpoint_changes(df, code, references, start, end)
    if 'AUS' not in changes.country_code.values:
        raise ValueError(f'Australia is missing an exact endpoint for {code}.')
    return changes


# Hold Australia fixed and resample comparator countries to measure composition sensitivity.
def bootstrap_oriented_gap(australia_change, comparator_changes):
    rng = np.random.default_rng(METHOD3_SEED)
    n = len(comparator_changes)
    gaps = np.empty(METHOD3_REPLICATES)
    for replicate in range(METHOD3_REPLICATES):
        resample = rng.choice(comparator_changes, size=n, replace=True)
        gaps[replicate] = australia_change - np.median(resample)
    return gaps


# Treat each eligible country as focal once to locate Australia in the same gap distribution.
def placebo_gap_summary(changes):
    focal_gaps = []
    for focal in changes.itertuples(index=False):
        others = changes.loc[changes.country_code.ne(focal.country_code), 'oriented_change']
        focal_gaps.append((focal.country_code, focal.oriented_change - others.median()))
    focal_gaps = pd.DataFrame(focal_gaps, columns=['country_code', 'focal_oriented_gap'])
    australia_gap = focal_gaps.loc[focal_gaps.country_code.eq('AUS'), 'focal_oriented_gap'].iloc[0]
    australia_rank = focal_gaps.focal_oriented_gap.rank(ascending=False, method='average').loc[focal_gaps.country_code.eq('AUS')].iloc[0]
    percentile = 100 * (len(focal_gaps) - australia_rank) / (len(focal_gaps) - 1)
    others = focal_gaps.loc[focal_gaps.country_code.ne('AUS'), 'focal_oriented_gap']
    return {
        'australia_gap': australia_gap,
        'percentile': percentile,
        'focal_country_count': len(focal_gaps),
        'more_favourable': int(others.gt(australia_gap).sum()),
        'less_favourable': int(others.lt(australia_gap).sum()),
        'tied': int(others.eq(australia_gap).sum()),
    }


# Build one validated bootstrap and placebo record for each primary outcome.
bootstrap_rows, placebo_rows = [], []
for spec, expected_count, expected_gap in zip(MATERIAL_SOCIAL_SPECS, EXPECTED_COUNTS, EXPECTED_GAPS):
    code, start, end, period = spec
    changes = exact_outcome_changes(code, start, end)
    australia = changes.loc[changes.country_code.eq('AUS')].iloc[0]
    comparators = changes.loc[changes.country_code.ne('AUS'), 'oriented_change'].to_numpy()
    if len(comparators) != expected_count:
        raise ValueError(f'{code} has {len(comparators)} comparators, expected {expected_count}.')
    primary = primary_endpoints.loc[primary_endpoints.indicator_code.eq(code)].iloc[0]
    observed_median = float(np.median(comparators))
    observed_gap = float(australia.oriented_change - observed_median)
    if not np.isclose(observed_gap, primary.australia_minus_comparator_median_oriented, atol=1e-10):
        raise ValueError(f'Method 3 does not reconcile with Method 2 for {code}.')
    if not np.isclose(observed_gap, expected_gap, atol=1e-5):
        raise ValueError(f'Unexpected observed oriented gap for {code}: {observed_gap}.')
    bootstrap_gaps = bootstrap_oriented_gap(float(australia.oriented_change), comparators)
    lower, upper = np.quantile(bootstrap_gaps, [0.025, 0.975])
    crosses_zero = bool(lower <= 0 <= upper)
    sensitivity = 'comparator-sensitive' if crosses_zero else 'comparatively stable'
    placebo = placebo_gap_summary(changes)
    if not np.isclose(placebo['percentile'], primary.australia_favourable_percentile, atol=1e-10):
        raise ValueError(f'Placebo percentile does not reconcile with Method 2 for {code}.')
    bootstrap_rows.append({
        'indicator_code': code, 'indicator': primary.indicator, 'unit': primary.unit,
        'comparison_period': period, 'australia_oriented_change': australia.oriented_change,
        'observed_comparator_median_oriented_change': observed_median, 'observed_oriented_gap': observed_gap,
        'bootstrap_gap_median': float(np.median(bootstrap_gaps)), 'bootstrap_ci_lower': float(lower),
        'bootstrap_ci_upper': float(upper), 'interval_crosses_zero': crosses_zero,
        'sensitivity_label': sensitivity, 'eligible_comparator_country_count': len(comparators),
        'bootstrap_replicates': METHOD3_REPLICATES, 'random_seed': METHOD3_SEED,
        'uncertainty_scope': UNCERTAINTY_SCOPE,
    })
    placebo_rows.append({
        'indicator_code': code, 'indicator': primary.indicator, 'unit': primary.unit,
        'comparison_period': period, 'australia_placebo_oriented_gap': placebo['australia_gap'],
        'australia_favourable_percentile': placebo['percentile'],
        'focal_country_count': placebo['focal_country_count'],
        'eligible_comparator_country_count': len(comparators),
        'more_favourable_country_count': placebo['more_favourable'],
        'less_favourable_country_count': placebo['less_favourable'],
        'tied_country_count': placebo['tied'],
        'percentile_definition': '0 = least favourable; 100 = most favourable; average ranks for ties',
        'evidence_relationship': EVIDENCE_RELATIONSHIP,
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)
placebo_results = pd.DataFrame(placebo_rows)
assert bootstrap_results.indicator_code.tolist() == OUTCOME_ORDER
assert placebo_results.indicator_code.tolist() == OUTCOME_ORDER
assert bootstrap_results.eligible_comparator_country_count.tolist() == EXPECTED_COUNTS
assert (bootstrap_results.bootstrap_ci_lower <= bootstrap_results.bootstrap_gap_median).all()
assert (bootstrap_results.bootstrap_gap_median <= bootstrap_results.bootstrap_ci_upper).all()
assert (placebo_results.more_favourable_country_count + placebo_results.less_favourable_country_count + placebo_results.tied_country_count == placebo_results.eligible_comparator_country_count).all()

# Persist both Method 3 tables and immediately verify their CSV round trips.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
bootstrap_path = TABLE_DIR / 'material_social_bootstrap_results.csv'
placebo_path = TABLE_DIR / 'material_social_placebo_results.csv'
bootstrap_results.to_csv(bootstrap_path, index=False)
placebo_results.to_csv(placebo_path, index=False)
pd.testing.assert_frame_equal(bootstrap_results, pd.read_csv(bootstrap_path), check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(placebo_results, pd.read_csv(placebo_path), check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
display(bootstrap_results.round(3))
display(placebo_results.round(3))


In [ ]:
# Separate axes preserve the native units: income is USD PPP per person; all
# other outcomes are percentage points. Values are not visually commensurate.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plot_table = bootstrap_results.merge(
    placebo_results[['indicator_code', 'australia_favourable_percentile']],
    on='indicator_code', validate='one_to_one'
)
display_names = {
    '1_1': 'Household income per person', '2_1': 'Employment rate',
    '7_1_DEP': 'Lack of social support', '11_2': 'Negative affect',
}
axis_units = {
    '1_1': 'USD per person, PPP', '2_1': 'Percentage points',
    '7_1_DEP': 'Percentage points', '11_2': 'Percentage points',
}
fig, axes = plt.subplots(4, 1, figsize=(10, 9), constrained_layout=True)
for ax, row in zip(axes, plot_table.itertuples(index=False)):
    left = row.observed_oriented_gap - row.bootstrap_ci_lower
    right = row.bootstrap_ci_upper - row.observed_oriented_gap
    ax.axvline(0, color='#555555', linewidth=1, zorder=0)
    ax.errorbar(row.observed_oriented_gap, 0, xerr=[[left], [right]], fmt='o',
                color='#D55E00', ecolor='#0072B2', elinewidth=3, capsize=4, markersize=8)
    ax.set_yticks([])
    ax.set_title(display_names[row.indicator_code], loc='left', fontweight='bold')
    ax.set_xlabel(f'Oriented Australia-minus-median gap ({axis_units[row.indicator_code]}; positive favours Australia)')
    ax.text(0.99, 0.82,
            f'Placebo percentile: {row.australia_favourable_percentile:.1f} | {row.eligible_comparator_country_count} comparators | {row.sensitivity_label}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9)
fig.suptitle('Australia’s common-endpoint gaps: comparator-country sensitivity', fontweight='bold')
figure_path = FIGURE_DIR / 'material_social_comparative_gaps.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
assert figure_path.exists() and figure_path.stat().st_size > 0
plt.show()
print(f'Wrote {figure_path.relative_to(PROJECT_ROOT)} at 300 dpi')
